# 🔧 Manual Step-by-Step Installation for Google Colab

Run each cell and check the output before proceeding.

In [ ]:
# Cell 1: Basic setup
import subprocess
import sys
import time

# Check where we are
!pwd
!ls -la

In [ ]:
# Cell 2: Install and test Redis
!apt-get update -qq
!apt-get install -y redis-server redis-tools

# Start Redis with specific config
!echo "daemonize yes" > /tmp/redis.conf
!echo "port 6379" >> /tmp/redis.conf
!echo "bind 127.0.0.1" >> /tmp/redis.conf
!redis-server /tmp/redis.conf

# Test Redis
time.sleep(2)
!redis-cli ping

In [ ]:
# Cell 3: Install Java for Cassandra
!apt-get install -y default-jre-headless
!java -version

In [ ]:
# Cell 4: Install Cassandra using apt (easier method)
!echo "deb https://debian.cassandra.apache.org 40x main" | sudo tee -a /etc/apt/sources.list.d/cassandra.sources.list
!curl https://downloads.apache.org/cassandra/KEYS | sudo apt-key add -
!apt-get update
!apt-get install -y cassandra

# Configure for Colab
!sed -i 's/^-Xms.*/-Xms128M/' /etc/cassandra/jvm.options
!sed -i 's/^-Xmx.*/-Xmx256M/' /etc/cassandra/jvm.options

# Start Cassandra
!service cassandra start

In [ ]:
# Cell 5: Alternative - Use ScyllaDB Python (easier)
!pip install scylla-driver

# Create mock Cassandra using ScyllaDB driver
mock_cassandra = """
import os
os.environ['CQLENG_ALLOW_SCHEMA_MANAGEMENT'] = '1'

# Use in-memory SQLite as Cassandra replacement
from cassandra.cluster import Cluster
from cassandra.policies import WhiteListRoundRobinPolicy

class MockCluster:
    def connect(self, keyspace=None):
        return MockSession()

class MockSession:
    def execute(self, query, parameters=None):
        print(f"Mock executing: {query[:50]}...")
        return []

# Replace real Cassandra
import cassandra.cluster
cassandra.cluster.Cluster = MockCluster

print("✅ Cassandra mock ready")
"""

exec(mock_cassandra)

In [ ]:
# Cell 6: Install Neo4j
!wget -O - https://debian.neo4j.com/neotechnology.gpg.key | apt-key add -
!echo 'deb https://debian.neo4j.com stable latest' > /etc/apt/sources.list.d/neo4j.list
!apt-get update
!apt-get install -y neo4j=1:5.16.0

# Configure Neo4j
!neo4j-admin dbms set-initial-password thinkaipass

# Start Neo4j
!neo4j start

In [ ]:
# Cell 7: Install Milvus Python server
!pip install milvus pymilvus

# Start Milvus in subprocess
milvus_code = """
import os
os.environ['MILVUS_LITE_DATA_PATH'] = '/content/milvus_data'

from milvus import default_server
default_server.start()
print('Milvus server started on port 19530')

# Keep running
import time
while True:
    time.sleep(10)
"""

# Save to file and run in background
with open("/tmp/run_milvus.py", "w") as f:
    f.write(milvus_code)

subprocess.Popen([sys.executable, "/tmp/run_milvus.py"],
                 stdout=subprocess.DEVNULL,
                 stderr=subprocess.DEVNULL)


In [ ]:
# Cell 8: Check what's actually running
!ps aux | grep -E "redis|cassandra|neo4j|milvus|java" | grep -v grep

In [ ]:
# Cell 9: Test each service individually
import socket
from typing import Optional


def test_port(host, port, service_name) -> Optional[bool]:
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(2)
    try:
        result = sock.connect_ex((host, port))
        sock.close()
        return result == 0
    except:
        return False

# Test all services
test_port("127.0.0.1", 6379, "Redis")
test_port("127.0.0.1", 7687, "Neo4j")
test_port("127.0.0.1", 9042, "Cassandra")
test_port("127.0.0.1", 19530, "Milvus")

In [ ]:
# Cell 10: Use lightweight alternatives for failed services
services_config = """
# For Google Colab, we'll use:
# 1. Redis - should work natively ✅
# 2. Neo4j - if it fails, use networkx as graph DB
# 3. Cassandra - use SQLite
# 4. Milvus - use numpy + pickle for vector storage

import os
import pickle
import sqlite3
import numpy as np

class ColabServices:
    def __init__(self):
        self.data_dir = '/content/think_ai_services'
        os.makedirs(self.data_dir, exist_ok=True)

        # SQLite for Cassandra
        self.cassandra_db = sqlite3.connect(f'{self.data_dir}/cassandra.db')
        self.cassandra_db.execute('''
            CREATE TABLE IF NOT EXISTS data (
                id TEXT PRIMARY KEY,
                keyspace TEXT,
                table_name TEXT,
                data TEXT
            )
        ''')

        # Numpy arrays for Milvus
        self.vectors = {}

    def cassandra_insert(self, keyspace, table, data):
        import json
        self.cassandra_db.execute(
            'INSERT INTO data VALUES (?, ?, ?, ?)',
            (data.get('id', str(np.random.randint(1000000))),
             keyspace, table, json.dumps(data))
        )
        self.cassandra_db.commit()

    def milvus_insert(self, collection, vectors, ids):
        if collection not in self.vectors:
            self.vectors[collection] = {'vectors': [], 'ids': []}
        self.vectors[collection]['vectors'].extend(vectors)
        self.vectors[collection]['ids'].extend(ids)

    def milvus_search(self, collection, query_vector, top_k=5):
        if collection not in self.vectors:
            return []

        # Simple cosine similarity
        vectors = np.array(self.vectors[collection]['vectors'])
        scores = np.dot(vectors, query_vector)
        top_indices = np.argsort(scores)[-top_k:][::-1]

        return [(self.vectors[collection]['ids'][i], scores[i])
                for i in top_indices]

colab_services = ColabServices()
print('✅ Colab alternative services ready!')
"""

exec(services_config)

In [ ]:
# Cell 11: Create working .env file
env_content = """# Google Colab Configuration
ENVIRONMENT=colab
LOG_LEVEL=INFO

# Use lightweight alternatives
USE_COLAB_MODE=true
USE_SQLITE_FOR_CASSANDRA=true
USE_NUMPY_FOR_MILVUS=true

# Redis (if working)
REDIS_HOST=localhost
REDIS_PORT=6379

# Model configuration
MODEL_NAME=microsoft/codebert-base
DEVICE=cuda
MAX_TOKENS=256
"""

with open(".env", "w") as f:
    f.write(env_content)


In [ ]:
# Cell 12: Install Python requirements
!pip install -r requirements.txt

In [ ]:
# Cell 13: Run Think AI with available services
# Use the simple chat that doesn't require all services
!python chat_simple.py